In [1]:
# Load required libraries

import boto3
import pandas as pd

In [2]:
# Downloaded the CFPB compliants data from this website on June 16, 2026
# https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data

In [3]:
# Either activate sampling to load the sampled data OR deactive sampling to load the full data

sampling = False

# In order to load the full data, I create and ran an AWS Cloud SageMaker Notebook (ml.m5.2xlarge) with a volume size of 50 GB EBS

In [4]:
# Import the raw CSV file of complaints from my AWS S3 bucket into a Pandas data frame

if sampling:
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket='aletheia-public', Key='complaints_16Jun2026_sampled.csv')
    df = pd.read_csv(obj['Body'])
else:
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket='aletheia-public', Key='complaints_16Jun2026.csv')
    df = pd.read_csv(obj['Body'])

/tmp/ipykernel_39702/3894271744.py:10: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(obj['Body'])


In [5]:
# The sampled dataset has 1,048,575 observations
# The full dataset has 15,896,496 observations

df.shape

(15896496, 16)

In [6]:
# List the raw data types of all columns of the Pandas data frame

df.dtypes

Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

In [7]:
# Convert the date columns from a string object format to a datetime format and then normalize the date to midnight, i.e. YYYY-MM-DD

df['Date received'] = pd.to_datetime(df['Date received'].str[:10]).dt.normalize()
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'].str[:10]).dt.normalize()

In [8]:
# Look at a few obs

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,CL Holdings LLC,CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907454
1,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Portfolio Recovery Associates, LLC",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907265
2,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Portfolio Recovery Associates, LLC",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907334
3,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company believes it acted appropriately as aut...,"Diversified Adjustment Service, Inc.",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907328
4,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Adler Wallach & Associates, Inc.",CA,90061,NaN,Web,2023-04-30,Closed with explanation,No,6907322


In [9]:
# Group complaints by year received and count observations

df.groupby(df['Date received'].dt.year).size()

Date received
2011       2536
2012      72368
2013     108214
2014     152909
2015     168273
2016     191294
2017     242747
2018     257133
2019     277240
2020     444214
2021     495942
2022     800257
2023    1292053
2024    2734271
2025    5443005
2026    3214040
dtype: int64

In [10]:
# Group complaints by product category and count observations

df.groupby(df['Product']).size()

Product
Bank account or service                                                            86198
Checking or savings account                                                       371944
Consumer Loan                                                                      31559
Credit card                                                                       319004
Credit card or prepaid card                                                       206356
Credit reporting                                                                  140426
Credit reporting or other personal consumer reports                             10495189
Credit reporting, credit repair services, or other personal consumer reports     2163774
Debt collection                                                                  1119431
Debt or credit management                                                           9110
Money transfer, virtual currency, or money service                                181171
Money transfe

In [11]:
# Group complaints by issue category and count observations

df.groupby(df['Issue']).size()

Issue
APR or interest rate                                       5506
Account opening, closing, or management                   37959
Account terms and changes                                   484
Adding money                                                202
Advertising                                                 570
                                                          ...  
Vehicle was repossessed or sold the vehicle                 684
Was approved for a loan, but didn't receive money           110
Was approved for a loan, but didn't receive the money       508
Written notification about debt                          218662
Wrong amount charged or received                           2025
Length: 178, dtype: int64

In [12]:
# Group by the tags and count obsevations

df.groupby(df['Tags']).size()

Tags
Older American                   221082
Older American, Servicemember     56421
Servicemember                    486732
dtype: int64

In [13]:
# Group by the company public response and count obsevations

df.groupby(df['Company public response']).size()

Company public response
Company believes complaint caused principally by actions of third party outside the control or direction of the company       8964
Company believes complaint is the result of an isolated error                                                                 7104
Company believes complaint relates to a discontinued policy or procedure                                                       130
Company believes complaint represents an opportunity for improvement to better serve consumers                                5906
Company believes it acted appropriately as authorized by contract or law                                                    232917
Company believes the complaint is the result of a misunderstanding                                                           14372
Company believes the complaint provided an opportunity to answer consumer's questions                                         7921
Company can't verify or dispute the facts in the complaint 

In [14]:
# Group by company response to consumer and count obsevations

df.groupby(df['Company response to consumer']).size()

Company response to consumer
Closed                               17611
Closed with explanation            9622917
Closed with monetary relief         209407
Closed with non-monetary relief    5380608
Closed with relief                    5304
Closed without relief                17867
In progress                         613684
Untimely response                    29075
dtype: int64

In [15]:
# Group by timely response to consumer complaints and count obsevations

df.groupby(df['Timely response?']).size()

Timely response?
No       100940
Yes    15795556
dtype: int64

In [16]:
# Count up the number of unique companies having consumer complaints

df['Company'].nunique()

7963

In [17]:
# Find the top 10 companies with the most consumer complaints

df['Company'].value_counts().head(10)

Company
TRANSUNION INTERMEDIATE HOLDINGS, INC.    4284244
EQUIFAX, INC.                             4169212
Experian Information Solutions Inc.       3768499
BANK OF AMERICA, NATIONAL ASSOCIATION      180503
WELLS FARGO & COMPANY                      169294
JPMORGAN CHASE & CO.                       168972
CAPITAL ONE FINANCIAL CORPORATION          162375
CITIBANK, N.A.                             135291
SYNCHRONY FINANCIAL                         79681
Block, Inc.                                 66225
Name: count, dtype: int64

In [18]:
# Print consumer complaints where the complaint narrative contains a specific keyword

keyword = 'Data Science'

with pd.option_context('display.max_colwidth', None):
    filtered_column = df.loc[df['Consumer complaint narrative'].str.contains(keyword, case=False, na=False), 'Consumer complaint narrative']
    print(filtered_column)

2368808                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 